# 02 · Mastodon conversation feature pipeline

From toot observations to one feature vector per conversation, where a
conversation is identified by `(instance_toot_found_id, conversation_id)`.

```
Toot observations
    ├─ Stage 1 · Clean and validate
    ├─ Stage 2 · Language detection             (FastText)
    ├─ Stage 3 · Toxicity scores                (Detoxify multilingual)
    │            └─ conversation-level toxicity aggregates  → label source for training
    ├─ Stage 4 · Sentiment scores               (XLM-RoBERTa)
    ├─ Stage 5 · Structural features            (length, unique authors, reaction time)
    ├─ Stage 6 · DeepWalk embeddings            (karateclub, pooled → 384-dim)
    ├─ Stage 7 · Author features                (pooled → 9-dim)
    └─ Stage 8 · Feature matrix                 (384 + 9 + 5 + 3 = 401-dim)
```

Inputs (from notebook 01): `mastodon_toots_object.csv`, `replies.csv`.
The FastText language-identification model `lid.176.bin` is expected in `models/` at the repository root.

Stage 6 uses `karateclub`, which needs older `networkx`/`numpy` versions; it can be
run in the separate environment from `requirements-deepwalk.txt`. Every stage saves
its output, and Stage 8 can reload them in a fresh kernel.

In [ ]:
import csv
import gc
from pathlib import Path

import matplotlib.pyplot as plt
import networkx as nx
import numpy as np
import pandas as pd
from tqdm import tqdm

## Configuration

In [ ]:
DATASET = "mastodon"
DATA_DIR = Path("../../data") / DATASET
FIGURES_DIR = Path("../../figures") / DATASET
FASTTEXT_MODEL_PATH = Path("../../models/lid.176.bin")

# Inputs
TOOTS_FILE = DATA_DIR / "mastodon_toots_object.csv"
REPLIES_PATH = DATA_DIR / "replies.csv"

# Outputs
OUT_LANG = DATA_DIR / "mastodon_toots_obj_with_lang.csv"
OUT_FILTERED = DATA_DIR / "mastodon_data.csv"
OUT_TOX = DATA_DIR / "mastodon_data_with_toxicity.csv"
OUT_CONV_TOX = DATA_DIR / "final_mastodon_nodes_conv_tox.csv"
OUT_TOOT_FEATURES = DATA_DIR / "mastodon_toot_features.parquet"
OUT_SENT_TOX = DATA_DIR / "mastodon_data_with_sent_and_tox.csv"
OUT_SENTIMENT = DATA_DIR / "conversations_aggregated_sentiment.csv"
OUT_STRUCT = DATA_DIR / "conversation_statistics_features.csv"
OUT_EMBEDDINGS = DATA_DIR / "conversation_embeddings.parquet"
OUT_AUTHOR_FEATURES = DATA_DIR / "authors_features_per_inst_conv.pkl"
OUT_FEATURES_ALL = DATA_DIR / "mastodon_conversation_features_all.parquet"
OUT_FEATURES_NPY = DATA_DIR / "mastodon_conversation_features_all.npy"
OUT_INDEX = DATA_DIR / "mastodon_index_conversation_features_all.parquet"

# Languages supported by both the sentiment model and Detoxify.
SUPPORTED_LANGUAGES = ["en", "es", "fr", "it", "pt"]
STRICT_LANGUAGE_FILTER = False  # True: drop a whole conversation if any toot is unsupported
FASTTEXT_CONFIDENCE = 0.5       # predictions below this confidence become "unknown"
FASTTEXT_CHUNKSIZE = 100_000

DETOXIFY_BATCH = 512
SENTIMENT_BATCH = 512
TOXICITY_THRESHOLDS = [0.3, 0.4, 0.5, 0.6, 0.7, 0.8, 0.9]

REACTION_TIME_CAP_PCT = 0.99    # drop reaction times above this percentile

DW_KWARGS = dict(
    walk_number=10,
    walk_length=60,
    dimensions=128,
    window_size=10,
    epochs=1,
    learning_rate=0.05,
    seed=42,
    workers=8,
)

# Feature matrices restricted to conversations with more than N toots.
LENGTH_FILTERS = [5, 10]

CONVERSATION_KEY = ["instance_toot_found_id", "conversation_id"]
TOOT_OBSERVATION_KEY = CONVERSATION_KEY + ["toot_id"]

FIGURES_DIR.mkdir(parents=True, exist_ok=True)

## Stage 1 · Clean and validate

Keep the columns needed downstream, normalise `root_toot` to a boolean, and drop
rows with missing keys or text.

In [ ]:
df = pd.read_csv(TOOTS_FILE, dtype=str, low_memory=False)
print(f"Loaded {len(df):,} rows")

KEEP_COLUMNS = [
    "created_at",
    "text",
    "author_id",
    "instance_toot_belong_id",
    "conversation_id",
    "toot_id",
    "instance_toot_found_id",
    "reblogs_count",
    "root_toot",
    # Account metadata, used for the author features.
    "followers_count",
    "following_count",
    "statuses_count",
]
missing = set(KEEP_COLUMNS) - set(df.columns)
if missing:
    raise ValueError(f"Missing required columns: {sorted(missing)}")
df = df[KEEP_COLUMNS].copy()

TRUE_VALUES = {True, 1, "1", "true", "True", "TRUE"}
FALSE_VALUES = {False, 0, "0", "false", "False", "FALSE"}


def normalize_bool(series):
    def convert(v):
        if pd.isna(v):
            return pd.NA
        if v in TRUE_VALUES:
            return True
        if v in FALSE_VALUES:
            return False
        return pd.NA
    return series.map(convert).astype("boolean")


df["root_toot"] = normalize_bool(df["root_toot"])
for col in TOOT_OBSERVATION_KEY:
    df[col] = df[col].astype(str).str.strip()

before = len(df)
df = df.dropna(subset=TOOT_OBSERVATION_KEY).reset_index(drop=True)
print(f"Dropped {before - len(df):,} rows with missing keys")

before = len(df)
df = df.dropna(subset=["text"]).reset_index(drop=True)
print(f"Dropped {before - len(df):,} rows with missing text")

print(f"Rows: {len(df):,} | conversations: {df[CONVERSATION_KEY].drop_duplicates().shape[0]:,} "
      f"| instances: {df['instance_toot_found_id'].nunique():,}")

## Stage 2 · Language detection

FastText assigns a language and confidence to each toot. Toots in unsupported
languages are dropped; the rest of their conversation is kept. Set
`STRICT_LANGUAGE_FILTER = True` to instead drop every conversation that contains
an unsupported toot.

In [ ]:
import fasttext

fasttext.FastText.eprint = lambda x: None
ft_model = fasttext.load_model(str(FASTTEXT_MODEL_PATH))


def detect_language_batch(texts):
    """(language codes, confidence scores) for a list of texts."""
    cleaned = [" ".join(str(t).split()) if pd.notna(t) else "" for t in texts]
    langs = ["unknown"] * len(cleaned)
    scores = [0.0] * len(cleaned)

    valid_idx = [i for i, s in enumerate(cleaned) if s]
    if valid_idx:
        labels, probs = ft_model.predict([cleaned[i] for i in valid_idx], k=1)
        for j, i in enumerate(valid_idx):
            label = labels[j][0].replace("__label__", "") if labels[j] else "unknown"
            prob = float(probs[j][0]) if len(probs[j]) else 0.0
            langs[i] = label if prob >= FASTTEXT_CONFIDENCE else "unknown"
            scores[i] = prob
    return langs, scores


all_langs, all_scores = [], []
texts = df["text"].tolist()
for start in tqdm(range(0, len(texts), FASTTEXT_CHUNKSIZE), desc="Language detection"):
    langs, scores = detect_language_batch(texts[start:start + FASTTEXT_CHUNKSIZE])
    all_langs.extend(langs)
    all_scores.extend(scores)

df["lang"] = all_langs
df["lang_score"] = all_scores
print(df["lang"].value_counts().head(15).to_string())

before_rows = len(df)
before_convs = df[CONVERSATION_KEY].drop_duplicates().shape[0]

if STRICT_LANGUAGE_FILTER:
    supported = set(SUPPORTED_LANGUAGES)
    ok = df.groupby(CONVERSATION_KEY)["lang"].transform(lambda s: set(s).issubset(supported))
    df = df[ok].copy()
else:
    df = df[df["lang"].isin(SUPPORTED_LANGUAGES)].copy()

print(f"Rows:          {before_rows:,} → {len(df):,}")
print(f"Conversations: {before_convs:,} → {df[CONVERSATION_KEY].drop_duplicates().shape[0]:,}")

df.to_csv(OUT_LANG, index=False, encoding="utf-8", quoting=csv.QUOTE_ALL)

### Keep complete conversations with at least two authors

A conversation observed on an instance without its root toot is incomplete and is
dropped. Conversations with a single author are dropped too.

In [ ]:
root_keys = df.loc[df["root_toot"].eq(True), CONVERSATION_KEY].drop_duplicates()
before = df[CONVERSATION_KEY].drop_duplicates().shape[0]
df = df.merge(root_keys, on=CONVERSATION_KEY, how="inner")
print(f"Conversations with root toot: {before:,} → {df[CONVERSATION_KEY].drop_duplicates().shape[0]:,}")

author_counts = df.groupby(CONVERSATION_KEY)["author_id"].nunique().reset_index(name="unique_author_count")
eligible = author_counts.loc[author_counts["unique_author_count"] > 1, CONVERSATION_KEY]
before = df[CONVERSATION_KEY].drop_duplicates().shape[0]
df = df.merge(eligible, on=CONVERSATION_KEY, how="inner")
print(f"Conversations with ≥ 2 authors: {before:,} → {df[CONVERSATION_KEY].drop_duplicates().shape[0]:,}")
print(f"Toot rows: {len(df):,}")

df.to_csv(OUT_FILTERED, index=False, encoding="utf-8", quoting=csv.QUOTE_ALL)

## Stage 3 · Toxicity

The multilingual Detoxify model scores each unique toot once; scores are joined
back to every observation of that toot.

In [ ]:
from detoxify import Detoxify

TOX_COLS = ["toxicity", "severe_toxicity", "obscene", "identity_attack", "insult", "threat"]


def batchify(lst, size):
    for i in range(0, len(lst), size):
        yield lst[i:i + size]


tox_model = Detoxify("multilingual")

unique_toots = df.drop_duplicates(subset=["toot_id"])[["toot_id", "text"]].copy()
texts = unique_toots["text"].tolist()
print(f"Unique toots to score: {len(unique_toots):,} (from {len(df):,} rows)")

all_tox = {k: [] for k in TOX_COLS}
for batch in tqdm(batchify(texts, DETOXIFY_BATCH), total=len(texts) // DETOXIFY_BATCH + 1, desc="Detoxify"):
    res = tox_model.predict(batch)
    for k in TOX_COLS:
        vals = res[k]
        all_tox[k].extend(vals.tolist() if hasattr(vals, "tolist") else list(vals))

for k in TOX_COLS:
    unique_toots[k] = all_tox[k]

df = df.merge(unique_toots[["toot_id"] + TOX_COLS], on="toot_id", how="left")

del tox_model
gc.collect()

print(f"Mean toxicity: {df['toxicity'].mean():.4f}")
print(f"Toxic toots (≥ 0.5): {(df['toxicity'] >= 0.5).sum():,} ({(df['toxicity'] >= 0.5).mean():.1%})")

df.to_csv(OUT_TOX, index=False, encoding="utf-8", quoting=csv.QUOTE_ALL)

### Conversation-level toxicity aggregates

For each conversation: summary statistics of toot toxicity, the toxicity of the
root toot, and, for several toot-level thresholds, the number and fraction of
toxic **replies** (root excluded).

The output, `final_mastodon_nodes_conv_tox.csv`, is the `--nodes_csv` input of
`train_DeTox-Fed.py`. Conversation labels are not fixed here: the training script derives
them from these aggregates, so the toot threshold, minimum number and minimum
fraction of toxic replies can be varied per experiment.

In [ ]:
toots_tox = df.copy()
toots_tox["root_toxicity"] = np.where(toots_tox["root_toot"], toots_tox["toxicity"], np.nan)


def severity_any(scores):
    """1 - Π(1 - p_i): probability that at least one toot is toxic."""
    s = scores.to_numpy(dtype=float)
    return 1.0 - np.prod(1.0 - s)


conv_tox = (
    toots_tox.groupby(CONVERSATION_KEY)
    .agg(
        mean_toxicity=("toxicity", "mean"),
        median_toxicity=("toxicity", "median"),
        min_toxicity=("toxicity", "min"),
        max_toxicity=("toxicity", "max"),
        n_posts=("toxicity", "size"),
        severity_any=("toxicity", severity_any),
        root_toxicity=("root_toxicity", "max"),
        n_roots=("root_toot", "sum"),
    )
    .reset_index()
)

replies = toots_tox[~toots_tox["root_toot"]].copy()
reply_agg = {"n_replies": ("toot_id", "size")}
for thr in TOXICITY_THRESHOLDS:
    suffix = str(thr).replace(".", "_")
    replies[f"is_toxic_{suffix}"] = replies["toxicity"] >= thr
    reply_agg[f"num_toxic_replies_{suffix}"] = (f"is_toxic_{suffix}", "sum")
    reply_agg[f"frac_toxic_replies_{suffix}"] = (f"is_toxic_{suffix}", "mean")

reply_stats = replies.groupby(CONVERSATION_KEY).agg(**reply_agg).reset_index()

conv_tox = conv_tox.merge(reply_stats, on=CONVERSATION_KEY, how="left", validate="one_to_one")
conv_tox.to_csv(OUT_CONV_TOX, index=False)
print(f"Saved {len(conv_tox):,} conversations → {OUT_CONV_TOX}")

## Stage 4 · Sentiment

`cardiffnlp/twitter-xlm-roberta-base-sentiment` (Barbieri et al., 2022) labels each
unique toot as Positive / Neutral / Negative. The signed score is the label
(+1 / 0 / −1) times the model confidence.

Per conversation, with toots in chronological order:

| Feature | Description |
|---|---|
| `sent_mean` | Mean sentiment |
| `sent_final` | Sentiment of the last toot |
| `sent_min` | Most negative sentiment |
| `neg_drift` | Sum of the decreases between consecutive toots (≤ 0) |
| `sent_volatility` | Standard deviation of sentiment |

In [ ]:
import torch
from transformers import pipeline

sentiment_pipe = pipeline(
    "text-classification",
    model="cardiffnlp/twitter-xlm-roberta-base-sentiment",
    tokenizer="cardiffnlp/twitter-xlm-roberta-base-sentiment",
    device=0 if torch.cuda.is_available() else -1,
)
LABEL_MAP = {"Positive": 1.0, "Neutral": 0.0, "Negative": -1.0}

unique_toots_sent = df.drop_duplicates(subset=["toot_id"])[["toot_id", "text"]].copy()
texts = unique_toots_sent["text"].tolist()

labels_list, scores_list = [], []
for batch in tqdm(batchify(texts, SENTIMENT_BATCH), total=len(texts) // SENTIMENT_BATCH + 1, desc="Sentiment"):
    results = sentiment_pipe(batch, truncation=True, max_length=512)
    labels_list.extend(r["label"] for r in results)
    scores_list.extend(r["score"] for r in results)

unique_toots_sent["sentiment_label"] = labels_list
unique_toots_sent["sentiment_score"] = scores_list
unique_toots_sent["sentiment_numeric"] = (
    unique_toots_sent["sentiment_label"].map(LABEL_MAP) * unique_toots_sent["sentiment_score"]
)

SENT_COLS = ["sentiment_label", "sentiment_score", "sentiment_numeric"]
df = df.merge(unique_toots_sent[["toot_id"] + SENT_COLS], on="toot_id", how="left")

(
    unique_toots_sent.merge(unique_toots[["toot_id"] + TOX_COLS], on="toot_id", how="left")
    [["toot_id"] + TOX_COLS + SENT_COLS]
    .to_parquet(OUT_TOOT_FEATURES, index=False)
)
df.to_csv(OUT_SENT_TOX, index=False, encoding="utf-8", quoting=csv.QUOTE_ALL)
print(df["sentiment_label"].value_counts())

In [ ]:
SENTIMENT_FEATURE_COLS = ["sent_mean", "sent_final", "sent_min", "neg_drift", "sent_volatility"]


def sentiment_features(group):
    s = group["sentiment_numeric"].values
    diffs = np.diff(s)
    return pd.Series({
        "sent_mean": float(np.mean(s)),
        "sent_final": float(s[-1]),
        "sent_min": float(np.min(s)),
        "neg_drift": float(diffs[diffs < 0].sum()) if len(diffs) > 0 else 0.0,
        "sent_volatility": float(np.std(s)) if len(s) > 1 else 0.0,
    })


sentiment_conv_df = (
    df.sort_values(CONVERSATION_KEY + ["created_at"])
    .groupby(CONVERSATION_KEY, sort=False)
    .apply(sentiment_features)
    .reset_index()
)
sentiment_conv_df["sentiment_features"] = sentiment_conv_df[SENTIMENT_FEATURE_COLS].values.tolist()
sentiment_conv_df.to_csv(OUT_SENTIMENT, index=False)

print(f"Sentiment features: {len(sentiment_conv_df):,} conversations")
sentiment_conv_df[SENTIMENT_FEATURE_COLS].describe().round(4)

## Stage 5 · Structural features

| Feature | Description |
|---|---|
| `conversation_length` | Number of toots |
| `unique_author_count` | Number of distinct authors |
| `median_reaction_time_sec` | Median time between a toot and a reply to it |

Reaction times use replies between different authors with a positive time
difference; values above the 99th percentile are dropped. Conversations without
such replies get 0.

In [ ]:
replies_df = pd.read_csv(
    REPLIES_PATH,
    dtype={c: "string" for c in ["instance_toot_found_id", "conversation_id", "toot_id_parent", "toot_id_child"]},
)
replies_df.columns = replies_df.columns.str.strip()
for c in ["instance_toot_found_id", "conversation_id", "toot_id_parent", "toot_id_child"]:
    replies_df[c] = replies_df[c].astype(str).str.strip()

# Keep only reply edges of the retained conversations.
replies_df = replies_df.merge(df[CONVERSATION_KEY].drop_duplicates(), on=CONVERSATION_KEY, how="inner")
print(f"Reply edges in scope: {len(replies_df):,}")

conv_length = df.groupby(CONVERSATION_KEY).size().reset_index(name="conversation_length")
conv_authors = df.groupby(CONVERSATION_KEY)["author_id"].nunique().reset_index(name="unique_author_count")

toot_times = df[CONVERSATION_KEY + ["toot_id", "created_at", "author_id"]].copy()
toot_times["created_at"] = pd.to_datetime(toot_times["created_at"], utc=True, errors="coerce")

merged = replies_df.merge(
    toot_times.rename(columns={"toot_id": "_id", "created_at": "child_time", "author_id": "child_author"}),
    left_on=CONVERSATION_KEY + ["toot_id_child"],
    right_on=CONVERSATION_KEY + ["_id"],
    how="inner",
).drop(columns=["_id"])
merged = merged.merge(
    toot_times[["instance_toot_found_id", "toot_id", "created_at", "author_id"]].rename(
        columns={"toot_id": "_id", "created_at": "parent_time", "author_id": "parent_author"}
    ),
    left_on=["instance_toot_found_id", "toot_id_parent"],
    right_on=["instance_toot_found_id", "_id"],
    how="inner",
).drop(columns=["_id"])

merged = merged[
    (merged["child_author"] != merged["parent_author"])
    & (merged["toot_id_child"] != merged["toot_id_parent"])
]
merged["reaction_time"] = (merged["child_time"] - merged["parent_time"]).dt.total_seconds()
merged = merged[merged["reaction_time"] > 0]
merged = merged[merged["reaction_time"] <= merged["reaction_time"].quantile(REACTION_TIME_CAP_PCT)]

reaction_time = (
    merged.groupby(CONVERSATION_KEY)["reaction_time"].median().reset_index(name="median_reaction_time_sec")
)

conv_struct = (
    conv_length
    .merge(conv_authors, on=CONVERSATION_KEY, how="inner")
    .merge(reaction_time, on=CONVERSATION_KEY, how="left")
)
conv_struct["median_reaction_time_sec"] = conv_struct["median_reaction_time_sec"].fillna(0.0)
conv_struct.to_csv(OUT_STRUCT, index=False)
conv_struct.describe().round(2)

## Stage 6 · DeepWalk conversation embeddings

One undirected reply graph per conversation; DeepWalk gives a 128-dimensional
embedding per toot, pooled with mean, sum and max into 384 dimensions.
Conversations whose reply graph has fewer than two nodes get a zero vector.

In [ ]:
from karateclub import DeepWalk

edge_lists = {
    key: list(zip(grp["toot_id_parent"], grp["toot_id_child"]))
    for key, grp in replies_df.groupby(CONVERSATION_KEY)
}
print(f"Conversation graphs to embed: {len(edge_lists):,}")

EMB_DIM = DW_KWARGS["dimensions"]
records = []

for (inst, conv), edges in tqdm(edge_lists.items(), desc="DeepWalk"):
    G = nx.Graph()
    G.add_edges_from(edges)
    G.remove_edges_from(nx.selfloop_edges(G))

    if G.number_of_nodes() < 2:
        records.append({"instance_toot_found_id": inst, "conversation_id": conv, "embedding": []})
        continue

    # karateclub requires consecutive integer node labels.
    G_int = nx.relabel_nodes(G, {n: i for i, n in enumerate(G.nodes())})
    model = DeepWalk(**DW_KWARGS)
    try:
        model.fit(G_int)
        emb = model.get_embedding()
    except Exception:
        emb = np.zeros((1, EMB_DIM), dtype=np.float32)

    records.append({"instance_toot_found_id": inst, "conversation_id": conv, "embedding": list(emb)})


def pool_embeddings(emb_list):
    if not emb_list:
        return np.zeros(EMB_DIM * 3, dtype=np.float32)
    X = np.stack(emb_list).astype(np.float32)
    return np.concatenate([X.mean(0), X.sum(0), X.max(0)])


emb_df = pd.DataFrame(records)
emb_df["embedding_pooled"] = emb_df["embedding"].apply(pool_embeddings)
emb_df = emb_df.drop(columns=["embedding"])
emb_df.to_parquet(OUT_EMBEDDINGS, index=False)
print(f"Saved {len(emb_df):,} embeddings → {OUT_EMBEDDINGS}")

## Stage 7 · Author features

Mastodon statuses include the author's account metadata (`followers_count`,
`following_count`, `statuses_count`). Each author is counted once per conversation,
and the three values are pooled over the conversation's authors with mean, sum and
max:

`[mean_followers, mean_following, mean_statuses, sum_followers, sum_following, sum_statuses, max_followers, max_following, max_statuses]`

In [ ]:
AUTHOR_FEATURE_COLS = ["followers_count", "following_count", "statuses_count"]

author_obs = df[CONVERSATION_KEY + ["author_id"] + AUTHOR_FEATURE_COLS].copy()
for col in AUTHOR_FEATURE_COLS:
    author_obs[col] = pd.to_numeric(author_obs[col], errors="coerce")

# An author posting several times in a conversation is counted once.
author_obs = author_obs.drop_duplicates(subset=CONVERSATION_KEY + ["author_id"], keep="first")


def pool_author_features(group):
    a = group[AUTHOR_FEATURE_COLS].fillna(0).to_numpy(dtype=np.float32)
    if len(a) == 0:
        return np.zeros(len(AUTHOR_FEATURE_COLS) * 3, dtype=np.float32)
    return np.concatenate([a.mean(0), a.sum(0), a.max(0)]).astype(np.float32)


authors_per_conv = (
    author_obs.groupby(CONVERSATION_KEY, sort=False)
    .apply(pool_author_features)
    .reset_index(name="author_features_pooled")
)
assert authors_per_conv["author_features_pooled"].apply(len).eq(9).all()

authors_per_conv.to_pickle(OUT_AUTHOR_FEATURES)
print(f"Author features: {len(authors_per_conv):,} conversations")

## Stage 8 · Feature matrix

```
embedding_pooled        384  DeepWalk (mean, sum, max)
author_features_pooled    9  author features (mean, sum, max)
sentiment_features        5  mean, final, min, neg_drift, volatility
structural                3  log1p(length), log1p(unique authors), log1p(reaction time)
---------------------------
total                   401
```

Saves the matrix (`.npy`), its row index (`.parquet`), and the per-conversation
table without the vector columns. The next cell reloads all stage outputs, so this
stage can run in a fresh kernel.

In [ ]:
emb_df = pd.read_parquet(OUT_EMBEDDINGS)
authors_per_conv = pd.read_pickle(OUT_AUTHOR_FEATURES)
sentiment_conv_df = pd.read_csv(OUT_SENTIMENT)
sentiment_conv_df["sentiment_features"] = sentiment_conv_df[SENTIMENT_FEATURE_COLS].values.tolist()
conv_struct = pd.read_csv(OUT_STRUCT)

In [ ]:
for d in [emb_df, authors_per_conv, sentiment_conv_df, conv_struct]:
    for c in CONVERSATION_KEY:
        d[c] = d[c].astype(str).str.strip()

df_final = (
    emb_df
    .merge(authors_per_conv[CONVERSATION_KEY + ["author_features_pooled"]], on=CONVERSATION_KEY, how="inner")
    .merge(sentiment_conv_df[CONVERSATION_KEY + ["sentiment_features"]], on=CONVERSATION_KEY, how="inner")
    .merge(conv_struct, on=CONVERSATION_KEY, how="inner")
)

df_final["log_conversation_length"] = np.log1p(df_final["conversation_length"])
df_final["log_unique_author_count"] = np.log1p(df_final["unique_author_count"])
df_final["log_reaction_time"] = np.log1p(df_final["median_reaction_time_sec"])

df_final["features"] = [
    np.concatenate([
        np.asarray(emb, dtype=np.float32),
        np.asarray(auth, dtype=np.float32),
        np.asarray(sent, dtype=np.float32),
        np.array([log_cl, log_ua, log_rt], dtype=np.float32),
    ])
    for emb, auth, sent, log_cl, log_ua, log_rt in zip(
        df_final["embedding_pooled"].values,
        df_final["author_features_pooled"].values,
        df_final["sentiment_features"].values,
        df_final["log_conversation_length"].values,
        df_final["log_unique_author_count"].values,
        df_final["log_reaction_time"].values,
    )
]

X = np.stack(df_final["features"].values).astype(np.float32)
df_index = df_final[CONVERSATION_KEY].copy()

np.save(OUT_FEATURES_NPY, X)
df_index.to_parquet(OUT_INDEX, index=False)
df_final.drop(columns=["embedding_pooled", "author_features_pooled", "features"]).to_parquet(OUT_FEATURES_ALL, index=False)
print(f"Feature matrix {X.shape} → {OUT_FEATURES_NPY}")

### Length-filtered feature matrices

Feature matrices restricted to conversations with more than `N` toots, used for
the conversation-length experiments.

In [ ]:
for min_len in LENGTH_FILTERS:
    keep = (df_final["conversation_length"] > min_len).to_numpy()
    np.save(DATA_DIR / f"mastodon_conversation_features_length_{min_len}.npy", X[keep])
    df_index[keep].to_parquet(DATA_DIR / f"mastodon_index_conversation_features_length_{min_len}.parquet", index=False)
    print(f"length > {min_len}: {int(keep.sum()):,} conversations")

## Descriptive plots

In [ ]:
df = pd.read_csv(OUT_SENT_TOX, low_memory=False)
conv_struct = pd.read_csv(OUT_STRUCT)

fig, axes = plt.subplots(2, 3, figsize=(16, 9))

lang_top = df["lang"].value_counts().head(15)
axes[0, 0].barh(lang_top.index[::-1], lang_top.values[::-1])
axes[0, 0].set_title("Language distribution (top 15)")
axes[0, 0].set_xlabel("Toot count")

axes[0, 1].hist(df["toxicity"], bins=50, edgecolor="none")
axes[0, 1].axvline(0.5, color="red", linestyle="--", label="threshold = 0.5")
axes[0, 1].set_title("Toxicity score distribution")
axes[0, 1].set_xlabel("Toxicity score")
axes[0, 1].legend()

sent_counts = df["sentiment_label"].value_counts()
axes[0, 2].bar(sent_counts.index, sent_counts.values)
axes[0, 2].set_title("Sentiment label distribution")

axes[1, 0].hist(conv_struct["conversation_length"], bins=50, edgecolor="none")
axes[1, 0].set_xlim(0, conv_struct["conversation_length"].quantile(0.99))
axes[1, 0].set_title("Conversation length")
axes[1, 0].set_xlabel("Number of toots")

axes[1, 1].hist(conv_struct["unique_author_count"], bins=30, edgecolor="none")
axes[1, 1].set_title("Unique authors per conversation")
axes[1, 1].set_xlabel("Author count")

rt = conv_struct["median_reaction_time_sec"]
axes[1, 2].hist(np.log1p(rt[rt > 0]), bins=50, edgecolor="none")
axes[1, 2].set_title("Median reaction time")
axes[1, 2].set_xlabel("log(1 + seconds)")

plt.tight_layout()
plt.savefig(FIGURES_DIR / "mastodon_pipeline_summary_plots.pdf", bbox_inches="tight", dpi=150)
plt.show()

### Toxic and non-toxic toots per instance

In [ ]:
plt.rcParams.update({
    "font.size": 16,
    "axes.titlesize": 20,
    "axes.labelsize": 20,
    "xtick.labelsize": 15,
    "ytick.labelsize": 20,
    "legend.fontsize": 20,
})

TOP_N = None  # all instances

df["is_toxic"] = df["toxicity"] >= 0.5

instance_toots = (
    df.groupby(["instance_toot_found_id", "is_toxic"])
    .size()
    .unstack(fill_value=0)
    .rename(columns={False: "Non-toxic", True: "Toxic"})
)
for col in ["Non-toxic", "Toxic"]:
    if col not in instance_toots.columns:
        instance_toots[col] = 0

instance_toots["Total"] = instance_toots["Non-toxic"] + instance_toots["Toxic"]
instance_toots = instance_toots.sort_values("Total", ascending=False)
if TOP_N is not None:
    instance_toots = instance_toots.head(TOP_N)

ax = instance_toots[["Non-toxic", "Toxic"]].plot(
    kind="bar", stacked=True, figsize=(14, 6), color=["#cce8ca", "pink"], width=0.8
)
ax.set_yscale("log")
ax.set_xlabel("Instance (id)")
ax.set_ylabel("Number of toots (log scale)")
ax.set_title(f"Mastodon | Number of Toots per Instance")
ax.tick_params(axis="x", rotation=70)
ax.legend(title=None, labels=["Non-toxic", "Toxic"])

# Percentage of toxic toots above each bar.
for i, (_, row) in enumerate(instance_toots.iterrows()):
    if row["Total"] > 0:
        ax.text(
            i, row["Total"] * 10, f"{100 * row['Toxic'] / row['Total']:.1f}%",
            ha="center", va="top", fontsize=17, color="black", rotation=40,
            bbox=dict(facecolor="pink", edgecolor="pink", alpha=0.9, boxstyle="round,pad=0.25"),
        )

ax.set_ylim(bottom=1, top=instance_toots["Total"].max() * 20)
plt.tight_layout()
plt.savefig(FIGURES_DIR / "mastodon_toots_per_instance.pdf", bbox_inches="tight")
plt.show()

### Distributions across instances and conversations

In [ ]:
def get_cdf(values):
    values = np.sort(np.asarray(values))
    return values, np.arange(1, len(values) + 1) / len(values)


counts = (
    df.groupby(["instance_toot_found_id", "is_toxic"])
    .size()
    .unstack(fill_value=0)
    .rename(columns={False: "Non-toxic", True: "Toxic"})
)

fig, ax = plt.subplots(figsize=(8, 6))
for col in ["Non-toxic", "Toxic"]:
    if col in counts.columns:
        ax.step(*get_cdf(counts[col].values), where="post", label=col)
ax.set_xscale("log")
ax.set_xlabel("Number of toots per instance", fontsize=20)
ax.set_ylabel("CDF", fontsize=20)
ax.tick_params(axis="both", labelsize=20)
ax.legend(fontsize=20)
ax.grid(alpha=0.3)
plt.tight_layout()
plt.savefig(FIGURES_DIR / "mastodon_toxic_non_toxic_toots_cdf.pdf", bbox_inches="tight")
plt.show()

In [ ]:
def plot_cdf(values, xlabel, title, filename):
    values = np.asarray(values, dtype=float)
    values = np.sort(values[np.isfinite(values)])
    y = np.arange(1, len(values) + 1) / len(values)

    plt.figure(figsize=(7, 5))
    plt.plot(values, y)
    plt.xscale("log")
    plt.xlabel(xlabel)
    plt.ylabel(r"$P(X \leq x)$ | Fraction")
    plt.title(title)
    plt.ylim(0, 1)
    plt.grid(alpha=0.3)
    plt.tight_layout()
    plt.savefig(FIGURES_DIR / filename, bbox_inches="tight")
    plt.show()


plot_cdf(
    df.groupby("instance_toot_found_id")["conversation_id"].nunique().values,
    "Number of conversations",
    "Mastodon | CDF of Conversations per Instance",
    "mastodon_conversations_per_instance_cdf.pdf",
)
plot_cdf(
    df.groupby(CONVERSATION_KEY)["toot_id"].nunique().values,
    "Conversation length (number of toots)",
    "Mastodon | CDF of Conversation Length",
    "mastodon_conversations_length_cdf.pdf",
)